In [1]:
import pandas as pd

# Carregar dataset
df = pd.read_excel("../dataset/dataset_velocidade_v2.xlsx")

# Extrair os 3 primeiros caracteres da velocidade
df['vel_prefixo'] = df['velocidade'].apply(lambda x: str(round(x, 5))[:4])  # 0.08 -> '0.08'

# Converter para float para usar como feature
df['vel_prefixo'] = df['vel_prefixo'].astype(float)

# Visualizar
print(df.head())


  movimento  tempo  velocidade  flag  vel_prefixo
0    avanco  1.230    0.081301     1         0.08
1    avanco  1.214    0.082372     1         0.08
2    avanco  1.213    0.082440     1         0.08
3    avanco  1.275    0.078431     0         0.07
4    avanco  1.405    0.071174     0         0.07


In [2]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['movimento_num'] = le.fit_transform(df['movimento'])
# 'avanco' -> 0, 'recuo' -> 1


In [3]:
df = pd.get_dummies(df, columns=['movimento'])
# Cria colunas 'movimento_avanco' e 'movimento_recuo' com 0/1


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Feature: prefixo da velocidade + movimento codificado
X = df[['vel_prefixo', 'movimento_num']]  # ou use as colunas dummies
y = df['flag']

# Separar treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Treinar modelo
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Avaliar
y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[583   0]
 [  0 255]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       583
           1       1.00      1.00      1.00       255

    accuracy                           1.00       838
   macro avg       1.00      1.00      1.00       838
weighted avg       1.00      1.00      1.00       838



In [5]:
import numpy as np
import pandas as pd

# Quantidade de amostras de teste
n_test = 20

# Gerar movimentos aleatórios
movimentos = np.random.choice(['avanco', 'recuo'], size=n_test)

# Gerar velocidades aleatórias:
# 70% normais (0.080 a 0.089), 30% anomalias (0.070 a 0.079)
velocidades = np.concatenate([
    np.random.uniform(0.080, 0.089, int(n_test*0.7)),
    np.random.uniform(0.070, 0.079, int(n_test*0.3))
])
np.random.shuffle(velocidades)

# Criar dataframe
df_test = pd.DataFrame({
    'movimento': movimentos,
    'velocidade': velocidades
})

# Codificar movimento (LabelEncoder)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_test['movimento_num'] = le.fit_transform(df_test['movimento'])

# Extrair prefixo da velocidade
df_test['vel_prefixo'] = df_test['velocidade'].apply(lambda x: float(str(round(x, 5))[:4]))

# Prever com modelo já treinado
y_pred = model.predict(df_test[['vel_prefixo', 'movimento_num']])
df_test['pred_flag'] = y_pred

# Mostrar resultado
print(df_test)


   movimento  velocidade  movimento_num  vel_prefixo  pred_flag
0     avanco    0.080398              0         0.08          1
1      recuo    0.081092              1         0.08          1
2      recuo    0.078042              1         0.07          0
3      recuo    0.084144              1         0.08          1
4      recuo    0.074643              1         0.07          0
5     avanco    0.083816              0         0.08          1
6      recuo    0.088481              1         0.08          1
7      recuo    0.083008              1         0.08          1
8     avanco    0.085336              0         0.08          1
9      recuo    0.081939              1         0.08          1
10    avanco    0.082961              0         0.08          1
11     recuo    0.075113              1         0.07          0
12     recuo    0.083673              1         0.08          1
13     recuo    0.087997              1         0.08          1
14    avanco    0.080025              0 